In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

env = gym.make("LunarLander-v2", render_mode="rgb_array")

obs, info = env.reset(seed=42)
img = env.render()

plt.imshow(img)
plt.show()


In [ ]:
def basic_policy(obs):
    if (obs[0] > 4):
        return 1
    if (obs[0] < -4):
        return 3
    return 0

totals = []
for episode in range(500):
    total_rewards = 0
    obs,info = env.reset(seed=episode)
    while True:
        action = basic_policy(obs)
        obs, reward, done, truncated, info = env.step(action)
        total_rewards += reward
        if done or truncated:
            break
    totals.append(total_rewards)
        

In [ ]:
print(np.mean(totals),np.std(totals),min(totals),max(totals))

In [ ]:
import torch
import torch.nn as nn

class PolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(8,10), nn.ReLU(), nn.Linear(10, 4))

    def forward(self, state):
        return self.net(state)

In [ ]:
def choose_action(model, obs):
    state = torch.as_tensor(obs)
    logit = model(state)
    dist = torch.distributions.Categorical(logits=logit)
    action = dist.sample()
    log_prob = dist.log_prob(action)
    return int(action.item()), log_prob

In [ ]:
def compute_returns(rewards, discount_factor):
    returns = rewards[:] #[:] copies the rewards
    for step in range(len(returns) - 1, 0, -1):
        returns[step -1] += returns[step] * discount_factor
        
    return torch.tensor(returns)

In [ ]:
def run_episode(model, env, seed=None):
    log_probs, rewards = [], []
    obs, info = env.reset(seed=seed)
    while True:
        action, log_prob = choose_action(model, obs)
        obs, reward, done, truncated, _info = env.step(action)
        log_probs.append(log_prob)
        rewards.append(reward)
        if (done or truncated):
            return log_probs, rewards

def train_reinforce(model, optimizer, env, n_episodes, discount_factor):
    for episode in range(n_episodes):
        seed = torch.randint(0, 2**32, size=()).item()
        log_probs, rewards = run_episode(model, env, seed=seed)
        returns = compute_returns(rewards, discount_factor)
        std_returns = (returns - returns.mean()) / (returns.std() + 1e-7)
        losses = [-logp * rt for logp, rt in zip(log_probs, std_returns)]
        loss = torch.stack(losses, dim=0).sum()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        print(f"\rEpisode{episode + 1}, Reward: {sum(rewards):.2f}", end=" ")

In [ ]:
torch.manual_seed(42)
model = PolicyNetwork()
optimizer = torch.optim.NAdam(model.parameters(), lr=0.06)
train_reinforce(model, optimizer, env, n_episodes=5, discount_factor=0.7)

In [ ]:
obs, info = env.reset(seed=42)


totals = []
for episode in range(1):
    total_rewards = 0
    obs,info = env.reset(seed=episode)
    while True:
        action = choose_action(model, obs)
        obs, reward, done, truncated, info = env.step(action[0])
        total_rewards += reward
        if done or truncated:
            break
    totals.append(total_rewards)
print(np.mean(totals),np.std(totals),min(totals),max(totals))

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(8, 48), nn.ReLU(), nn.Linear(48,48), nn.ReLU())

        self.actor_head = nn.Linear(48, 4)
        self.critic_head = nn.Linear(48, 1)

    def forward(self, state):
        features = self.body(state)
        return self.actor_head(features), self.critic_head(features).squeeze(-1)

In [ ]:
def choose_action_and_evaluate(model, obs):
    state = torch.as_tensor(obs)
    logit, state_value = model(state)
    dist = torch.distributions.Categorical(logits=logit)
    entropy = dist.entropy()
    action = dist.sample()
    log_prob = dist.log_prob(action)
    return int(action.item()), log_prob, state_value, entropy

In [ ]:
def ac_training_step(optimizer, criterion, state_value, target_value, log_prob, entropy, critic_weight=0.5, entropy_weight=0.0005):
    td_error = target_value - state_value
    actor_loss = -log_prob * td_error.detach() - entropy * entropy_weight
    critic_loss = criterion(state_value, target_value)
    loss = actor_loss + critic_weight * critic_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
def get_target_value(model, next_obs, reward, done, truncated, discount_factor):
    with torch.inference_mode():
        _, _, next_state_value, _ = choose_action_and_evaluate(model, next_obs)

    running = 0.0 if (done or truncated) else 1.0
    target_value = reward + running * discount_factor * next_state_value
    return target_value

In [ ]:
def run_episode_and_train(model, optimizer, criterion, env, discount_factor, critic_weight, seed=None):
    obs, _info = env.reset(seed=seed)
    total_rewards = 0
    while True:
        action, log_prob, state_value, entropy = choose_action_and_evaluate(model, obs)
        next_obs, reward, done, truncated, _info = env.step(action)
        target_value = get_target_value(model, next_obs, reward, done, truncated, discount_factor)
        ac_training_step(optimizer, criterion, state_value, target_value, log_prob, entropy, critic_weight)
        total_rewards += reward
        if  done or truncated:
            return total_rewards
        obs = next_obs

In [ ]:
def train_actor_critic(model, optimizer, criterion, env, n_episodes=400, discount_factor=0.95, critic_weight=0.3):
    totals = []
    model.train()
    best_avg = -float("inf")
    for episode in range(n_episodes):
        seed = torch.randint(0, 2**32, size=()).item()
        total_rewards = run_episode_and_train(model, optimizer, criterion, env, discount_factor, critic_weight, seed=seed)
        totals.append(total_rewards)

        if len(totals) >= 100:
            avg100 = np.mean(totals[-100:])
            if avg100 > best_avg:
                best_avg = avg100
                torch.save(model.state_dict(), "best_actor_critic.pth")
                
        print(f"\rEpisode: {episode + 1}, Rewards: {total_rewards}", end=" ")
    return totals

In [ ]:
torch.manual_seed(42)
ac_model = ActorCritic()
optimizer = torch.optim.NAdam(ac_model.parameters(), lr=1e-4)
criterion = nn.MSELoss()
totals = train_actor_critic(ac_model, optimizer, criterion, env, n_episodes=4000, critic_weight=.5, discount_factor=.99)
print("\n")
print(np.mean(totals),np.std(totals),min(totals),max(totals))

In [ ]:
import matplotlib.pyplot as plt

window = 100
moving_avg = np.convolve(totals, np.ones(window)/window, mode="valid")

plt.plot(totals, alpha=0.3, label="Raw rewards")
plt.plot(range(window-1, len(totals)), moving_avg, label="Moving avg (100)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Training Performance")
plt.legend()
plt.show()


totals = []
for episode in range(200):
    total_rewards = 0
    obs,info = env.reset(seed=episode)
    while True:
        action = choose_action_and_evaluate(ac_model, obs)
        obs, reward, done, truncated, info = env.step(action[0])
        total_rewards += reward
        if done or truncated:
            break
    totals.append(total_rewards)
print(np.mean(totals),np.std(totals),min(totals),max(totals))

In [ ]:
ac_model = ActorCritic()
ac_model.load_state_dict(torch.load("best_actor_critic.pth"))

totals = []
for episode in range(200):
    total_rewards = 0
    obs,info = env.reset(seed=episode)
    while True:
        action = choose_action_and_evaluate(ac_model, obs)
        obs, reward, done, truncated, info = env.step(action[0])
        total_rewards += reward
        if done or truncated:
            break
    totals.append(total_rewards)
print(np.mean(totals),np.std(totals),min(totals),max(totals))


window = 100
moving_avg = np.convolve(totals, np.ones(window)/window, mode="valid")

plt.plot(totals, alpha=0.3, label="Raw rewards")
plt.plot(range(window-1, len(totals)), moving_avg, label="Moving avg (100)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Training Performance")
plt.legend()
plt.show()
